# Settimana 9 — De Moivre, Laplace e la campana delle somme: il teorema del limite centrale

Questo è il notebook della Settimana 9 del corso di Probabilità e Statistica. Contiene gli script Python della [dispensa della settimana](https://qwertyjacob.github.io/PSI-VA-2627/settimana_09/), nello stesso ordine in cui compaiono: ogni script è una cella di codice, preceduta dal titolo della sezione in cui si trova. Le spiegazioni restano nella dispensa.

Per eseguire una cella, premi il pulsante ▶ alla sua sinistra. Gli script sono indipendenti l'uno dall'altro, quindi puoi eseguirli in qualsiasi ordine e modificarli liberamente. Colab non salva le modifiche nel notebook originale: se vuoi conservarle, scegli *File → Salva una copia in Drive*.

## Laboratorio: la campana delle somme

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_09/#lab-somme)

In [ ]:
from math import comb, factorial, sqrt
from statistics import NormalDist

import matplotlib.pyplot as plt
import numpy as np

Z = NormalDist()   # normale standard: Z.cdf(z) è la funzione Phi(z)


def cdf_uniformi(x, n):
    """P(S_n <= x) esatta per la somma di n uniformi tra 0 e 1."""
    tot = sum((-1) ** k * comb(n, k) * (x - k) ** n
              for k in range(int(x) + 1))
    return tot / factorial(n)


# P(S_n < n/3): valore esatto e normale con media n/2, varianza n/12.
print("  n   esatta   normale")
for n in [2, 5, 10]:
    esatta = cdf_uniformi(n / 3, n)
    normale = Z.cdf((n / 3 - n / 2) / sqrt(n / 12))
    print(f"{n:3d}   {esatta:.4f}   {normale:.4f}")

# Dieci dadi: la distribuzione esatta della somma si costruisce
# aggiungendo un dado alla volta (np.convolve somma su tutti i modi).
dado = np.ones(6) / 6
somma = np.array([1.0])
for _ in range(10):
    somma = np.convolve(somma, dado)
valori = np.arange(10, 61)               # la somma va da 10 a 60
vince = (valori <= 25) | (valori >= 45)
mu, sigma = 35, sqrt(10 * 35 / 12)
senza = 2 * (1 - Z.cdf(10 / sigma))      # soglie 25 e 45
con = 2 * (1 - Z.cdf(9.5 / sigma))       # soglie 25,5 e 44,5
print(f"\nDieci dadi: esatta {somma[vince].sum():.4f}   "
      f"normale {senza:.4f}   con la correzione {con:.4f}")

# Istogrammi della somma di n uniformi, con la campana sovrapposta.
rng = np.random.default_rng(1733)
fig, assi = plt.subplots(1, 4, figsize=(12, 3))
for asse, n in zip(assi, [1, 2, 5, 10]):
    s = rng.random((50_000, n)).sum(axis=1)
    campana = NormalDist(n / 2, sqrt(n / 12))
    x = np.linspace(s.min(), s.max(), 200)
    asse.hist(s, bins=50, density=True, alpha=0.6)
    asse.plot(x, [campana.pdf(v) for v in x])
    asse.set_title(f"n = {n}")
plt.tight_layout()
plt.show()

## Laboratorio: lampadine ed esecuzioni simulate

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_09/#lab-esecuzioni)

In [ ]:
from math import sqrt
from statistics import NormalDist

import numpy as np

rng = np.random.default_rng(1810)
Z = NormalDist()
prove = 10_000

# Le lampadine: la somma di 40 durate esponenziali con media 10.
durate = rng.exponential(10, size=(prove, 40)).sum(axis=1)
tlc = 1 - Z.cdf((365 - 400) / sqrt(4000))
print(f"Lampadine: simulata {np.mean(durate >= 365):.3f}   "
      f"TLC {tlc:.3f}")

# Il valore z con Phi(z) = 0,975 e il numero di esecuzioni.
z = Z.inv_cdf(0.975)
print(f"z = {z:.3f}   n >= {(z * 2 / 0.5) ** 2:.2f}")

# Tempi asimmetrici: 8 s più un'attesa esponenziale con media 2 s.
# Media 10 s, varianza 4 s², come nel problema delle slide.
for n in [10, 62, 320]:
    tempi = 8 + rng.exponential(2, size=(prove, n))
    medie = tempi.mean(axis=1)
    dentro = np.mean(np.abs(medie - 10) < 0.5)
    attesa = 2 * Z.cdf(0.5 / (2 / sqrt(n))) - 1
    print(f"n = {n:3d}: media entro 0,5 s nel {dentro:.1%} "
          f"dei casi (TLC {attesa:.1%})")

## Laboratorio: quando la campana non arriva

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_09/#lab-code)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(1853)


def pareto(dimensione):
    """Pareto con alfa = 1,5: P(X > x) = x**(-1,5) per x >= 1."""
    return (1 - rng.random(dimensione)) ** (-1 / 1.5)


# 1. Le code della somma standardizzata di n esponenziali.
#    Per una normale sarebbero entrambe il 2,28%.
print("  n   P(Z > 2)   P(Z < -2)")
for n in [1, 5, 30, 100]:
    s = rng.exponential(1, size=(20_000, n)).sum(axis=1)
    z = (s - n) / np.sqrt(n)
    print(f"{n:3d}    {np.mean(z > 2):.4f}     {np.mean(z < -2):.4f}")

# 2. Medie progressive di 100 000 osservazioni.
n = 100_000
k = np.arange(1, n + 1)
dati = {"esponenziale": rng.exponential(1, n),
        "Pareto 1,5": pareto(n),
        "Cauchy": rng.standard_cauchy(n)}
print("\nmedia dopo      100     1000    10000   100000")
for nome, x in dati.items():
    medie = np.cumsum(x) / k
    tappe = [medie[t - 1] for t in [100, 1_000, 10_000, n]]
    print(f"{nome:12s}", "".join(f"{m:9.3f}" for m in tappe))
    plt.plot(k, medie, label=nome)
plt.xscale("log")
plt.ylim(-3, 6)
plt.xlabel("numero di osservazioni")
plt.ylabel("media progressiva")
plt.legend()
plt.show()

# 3. La media di 1000 osservazioni, ripetuta 2000 volte: quanto
#    spesso cade lontano dal centro della distribuzione?
print()
for nome, x, centro, eps in [
        ("esponenziale", rng.exponential(1, (2000, 1000)), 1, 0.1),
        ("Pareto 1,5", pareto((2000, 1000)), 3, 0.3),
        ("Cauchy", rng.standard_cauchy((2000, 1000)), 0, 1)]:
    lontane = np.mean(np.abs(x.mean(axis=1) - centro) > eps)
    print(f"{nome:12s} media fuori da {centro} ± {eps}: {lontane:.1%}")